# BERTScore (semantic evaluation) for baseline vs BRIO checkpoints

BERTScore compares the meaning of a summary with the reference summary using BERT embeddings, so it complements ROUGE (word overlap).

- **Part A** runs on the 100 articles used for the Gemini judge. Quick.
- **Part B** generates summaries for the whole test set (10,972 articles) and scores them. This is the version to report in the paper.

Several BRIO checkpoints can be evaluated in one run: list them in `BRIO_CKPTS` in the Paths cell. They are processed one after another, and each one is compared with the baseline.

In [1]:
# Install condacolab. The runtime restarts after this cell, that is normal.
!pip install -q condacolab
import condacolab
condacolab.install()


📢 Announcement 📢
condacolab==0.2 will be released soon! Try it with:

    !pip install -q https://github.com/conda-incubator/condacolab/archive/main.zip
    import condacolab
    condacolab.install()

0.2.x introduces a new installation method based on Pixi, with customizable Python versions.
This may be breaking for your workflow. If that's the case, please report it at
https://github.com/conda-incubator/condacolab and pin your `pip install` command to:

    !pip install -q 'condacolab==0.1.*'

⏬ Downloading https://github.com/conda-forge/miniforge/releases/download/26.3.2-3/Miniforge3-26.3.2-3-Linux-x86_64.sh...
📦 Installing...
📌 Adjusting configuration...
🩹 Patching environment...
⏲ Done in 0:00:06
🔁 Restarting kernel...


### Env

In [1]:
# Clone the repo and mount Google Drive.
!git clone https://github.com/alvrian/BRIO.git

from google.colab import drive
drive.mount('/content/drive', force_remount=False)

%cd BRIO

Cloning into 'BRIO'...
remote: Enumerating objects: 518, done.
remote: Counting objects: 100% (227/227), done.
remote: Compressing objects: 100% (144/144), done.
remote: Total 518 (delta 162), reused 121 (delta 82), pack-reused 291 (from 2)
Receiving objects: 100% (518/518), 16.03 MiB | 13.37 MiB/s, done.
Resolving deltas: 100% (316/316), done.
Mounted at /content/drive
/content/BRIO


In [2]:
# Same env setup as BRIO_1_1.ipynb, plus the bert-score package and the fine-tuned IndoBART baseline.
! conda create --name env --file spec-file.txt
! conda run --no-capture-output -n env pip install -r requirements.txt
! conda run -n env python -c "import nltk;nltk.download('punkt')"
! git clone https://github.com/neulab/compare-mt.git
! cd compare-mt && conda run --no-capture-output -n env pip install -r requirements.txt
! cd compare-mt && conda run -n env python setup.py install
! conda run --no-capture-output -n env pip install bert-score
print("done setting up env")
! cp /content/drive/MyDrive/BRIO/indobart-liputan6-finetuned.zip .
! unzip -q indobart-liputan6-finetuned.zip
! rm indobart-liputan6-finetuned.zip
print("all done")

Streaming output truncated to the last 5000 lines.







cudatoolkit-11.1.1   | :  23% 0.23377823660513725/1 [00:13<00:47, 62.06s/it]










cudatoolkit-11.1.1   | :  24% 0.23542688425397462/1 [00:13<00:48, 63.40s/it]










cudatoolkit-11.1.1   | :  24% 0.23702480428284775/1 [00:13<00:48, 64.20s/it]










cudatoolkit-11.1.1   | :  24% 0.23859736050173877/1 [00:14<00:54, 71.17s/it]










cudatoolkit-11.1.1   | :  24% 0.2400430976707192/1 [00:14<00:53, 70.87s/it] 










cudatoolkit-11.1.1   | :  24% 0.241564926269646/1 [00:14<00:52, 69.38s/it] 










cudatoolkit-11.1.1   | :  24% 0.24339112058835816/1 [00:14<00:49, 65.56s/it]










cudatoolkit-11.1.1   | :  25% 0.2451919510970882/1 [00:14<00:47, 62.69s/it] 










cudatoolkit-11.1.1   | :  25% 0.24684059874592554/1 [00:14<00:46, 62.11s/it]










cudatoolkit-11.1.1   | :  25% 0.2484638825847808/1 [00:14<00:51, 68.66s/it] 










cudatoolkit-11.1.1   | :  25% 0.2502013035685555/1 [00:14<00:49, 65.43s/i

### Paths

In [3]:
# List every model_generation.bin saved on Drive, to copy the right paths into the next cell.
import glob
for p in sorted(glob.glob("/content/drive/MyDrive/BRIO/checkpoints/*/*/model_generation.bin")):
    print(p)

/content/drive/MyDrive/BRIO/checkpoints/BRIO_Liputan6_checkpoint_0-1/4000/model_generation.bin
/content/drive/MyDrive/BRIO/checkpoints/BRIO_Liputan6_checkpoint_0-1_second_seed/master_1_0/model_generation.bin
/content/drive/MyDrive/BRIO/checkpoints/BRIO_Liputan6_checkpoint_0-1_third_seed/master_1_0/model_generation.bin
/content/drive/MyDrive/BRIO/checkpoints/BRIO_Liputan6_checkpoint_1-1/4000/model_generation.bin
/content/drive/MyDrive/BRIO/checkpoints/BRIO_Liputan6_checkpoint_1-1/8000/model_generation.bin
/content/drive/MyDrive/BRIO/checkpoints/BRIO_Liputan6_checkpoint_1-100_12/master_1_100/model_generation.bin
/content/drive/MyDrive/BRIO/checkpoints/BRIO_Liputan6_checkpoint_1-100_8/10000/model_generation.bin
/content/drive/MyDrive/BRIO/checkpoints/BRIO_Liputan6_checkpoint_1-100_8/12000/model_generation.bin
/content/drive/MyDrive/BRIO/checkpoints/BRIO_Liputan6_checkpoint_1-100_8/2000/model_generation.bin
/content/drive/MyDrive/BRIO/checkpoints/BRIO_Liputan6_checkpoint_1-100_8/4000/model

In [6]:
import os

# Liputan6 canonical test set on Drive.
TEST_SOURCE = "/content/drive/MyDrive/BRIO/liputan6_candidates/canonical/test.source"
TEST_TARGET = "/content/drive/MyDrive/BRIO/liputan6_candidates/canonical/test.target"

# Fine-tuned IndoBART baseline, unzipped by the env cell.
BASELINE = "./indobart-liputan6-finetuned"

# BRIO checkpoints to evaluate: short name -> path of model_generation.bin.
# EDIT: add or remove lines. Keep the name "brio" for the checkpoint you used with the Gemini judge,
# so its saved summaries are reused.
CKPT_DIR = "/content/drive/MyDrive/BRIO/checkpoints"
# BRIO_CKPTS = {
#     "brio": CKPT_DIR + "/BRIO_Liputan6_checkpoint_1-1_second_seed/master_1_1/model_generation.bin",
#     "brio_seed3": CKPT_DIR + "/BRIO_Liputan6_checkpoint_1-1_third_seed/master_1_1/model_generation.bin",
#     "brio_cand8": CKPT_DIR + "/BRIO_Liputan6_checkpoint_1-1_8/master_1_1/model_generation.bin",
#     "brio_cand12": CKPT_DIR + "/BRIO_Liputan6_checkpoint_1-1_12/master_1_1/model_generation.bin",
# }

BRIO_CKPTS = {
    "brio_1_1_seed1": CKPT_DIR +  "/BRIO_Liputan6_checkpoint_1-1/8000/model_generation.bin",
    "brio_1_1_seed2":   CKPT_DIR + "/BRIO_Liputan6_checkpoint_1-1_second_seed/master_1_1/model_generation.bin",
    "brio_1_1_seed3":   CKPT_DIR + "/BRIO_Liputan6_checkpoint_1-1_third_seed/master_1_1/model_generation.bin",
    "brio_10_1_seed1":  CKPT_DIR +  "/BRIO_Liputan6_checkpoint_10-1-reduceLR/8000/model_generation.bin",
    "brio_10_1_seed2":  CKPT_DIR +  "/BRIO_Liputan6_checkpoint_10-1_second_seed/master_1_10/model_generation.bin",
    "brio_10_1_seed3":  CKPT_DIR +  "/BRIO_Liputan6_checkpoint_10-1_third_seed/master_1_10/model_generation.bin",
    "brio_100_1_seed1": CKPT_DIR +  "/BRIO_Liputan6_checkpoint_100-1-reduceLR/4000/model_generation.bin",
    "brio_100_1_seed2": CKPT_DIR +  "/BRIO_Liputan6_checkpoint_100-1-second_seed/master_1_100/model_generation.bin",
    "brio_100_1_seed3": CKPT_DIR +  "/BRIO_Liputan6_checkpoint_100-1-third_seed/master_1_100/model_generation.bin",
    "brio_1_1_seed1_12cand" : CKPT_DIR + '/BRIO_Liputan6_checkpoint_1-1_12/master_1_1/model_generation.bin',
    "brio_1_1_seed2_12cand" : CKPT_DIR + '/BRIO_Liputan6_checkpoint_1-1_12_second_seed/master_1_1/model_generation.bin',
    "brio_10_1_seed1_12cand" : CKPT_DIR + '/BRIO_Liputan6_checkpoint_1-10_12/master_1_10/model_generation.bin',
    "brio_10_1_seed2_12cand" : CKPT_DIR + '/BRIO_Liputan6_checkpoint_1-10_12_second_seed/master_1_10/model_generation.bin',
    # "brio_10_1_seed3_12cand" : CKPT_DIR + '/BRIO_Liputan6_checkpoint_1-10_12_third_seed/master_1_10/model_generation.bin',
    "brio_100_1_seed3_12cand" : CKPT_DIR + '/BRIO_Liputan6_checkpoint_1-100_12/master_1_100/model_generation.bin',
    "brio_1_1_seed1_8cand" : CKPT_DIR +'/BRIO_Liputan6_checkpoint_1-1_8/master_1_1/model_generation.bin',
    "brio_10_1_seed1_8cand" : CKPT_DIR + '/BRIO_Liputan6_checkpoint_1-10_8/master_1_10/model_generation.bin',
    "brio_100_1_seed1_8cand" : CKPT_DIR + '/BRIO_Liputan6_checkpoint_1-100_8/master_1_100/model_generation.bin'
}

# Folder with the 100-article sample (Part A) and a folder for the whole test set (Part B).
SAMPLE_DIR = "/content/drive/MyDrive/BRIO/eval_outputs"
FULL_DIR = "/content/drive/MyDrive/BRIO/eval_outputs_full"

for p in [TEST_SOURCE, TEST_TARGET, BASELINE] + list(BRIO_CKPTS.values()):
    print("found  " if os.path.exists(p) else "MISSING", p)

# Build the system list and the baseline-vs-checkpoint pairs once, used by every scoring cell.
SYSTEMS = " ".join(["baseline"] + list(BRIO_CKPTS))
PAIRS = " ".join("--pair baseline " + name for name in BRIO_CKPTS)
print(SYSTEMS)

found   /content/drive/MyDrive/BRIO/liputan6_candidates/canonical/test.source
found   /content/drive/MyDrive/BRIO/liputan6_candidates/canonical/test.target
found   ./indobart-liputan6-finetuned
found   /content/drive/MyDrive/BRIO/checkpoints/BRIO_Liputan6_checkpoint_1-1/8000/model_generation.bin
found   /content/drive/MyDrive/BRIO/checkpoints/BRIO_Liputan6_checkpoint_1-1_second_seed/master_1_1/model_generation.bin
found   /content/drive/MyDrive/BRIO/checkpoints/BRIO_Liputan6_checkpoint_1-1_third_seed/master_1_1/model_generation.bin
found   /content/drive/MyDrive/BRIO/checkpoints/BRIO_Liputan6_checkpoint_10-1-reduceLR/8000/model_generation.bin
found   /content/drive/MyDrive/BRIO/checkpoints/BRIO_Liputan6_checkpoint_10-1_second_seed/master_1_10/model_generation.bin
found   /content/drive/MyDrive/BRIO/checkpoints/BRIO_Liputan6_checkpoint_10-1_third_seed/master_1_10/model_generation.bin
found   /content/drive/MyDrive/BRIO/checkpoints/BRIO_Liputan6_checkpoint_100-1-reduceLR/4000/model_gener

In [7]:
# Same helper as before, with the batch size as a parameter.
def generate_all(out_dir, sample_n, batch_size=8):
    models = {"baseline": BASELINE}
    models.update(BRIO_CKPTS)
    for name, path in models.items():
        print("=====", name)
        !conda run --no-capture-output -n env python generate_outputs.py --name {name} --model_path "{path}" --src_file "{TEST_SOURCE}" --tgt_file "{TEST_TARGET}" --out_dir "{out_dir}" --sample_n {sample_n} --batch_size {batch_size}


### Part A: the 100 sampled articles

In [ ]:
# Summaries for the 100 sampled articles (only the checkpoints not saved yet are generated).
generate_all(SAMPLE_DIR, 1)

In [ ]:
# BERTScore for every system on the 100 articles. Results are saved in SAMPLE_DIR/bertscore/.
!conda run --no-capture-output -n env python bertscore_eval.py --data_dir "{SAMPLE_DIR}" --systems {SYSTEMS} {PAIRS}

### Part B: whole test set

Generation is the slow part and the time grows with every checkpoint in the list. If Colab disconnects, run the same cell again: finished models are skipped and the interrupted one continues from its last saved summary.

In [ ]:
# Summaries for all 10,972 test articles, for the baseline and every checkpoint.
# generate_all(FULL_DIR, 0)
generate_all(FULL_DIR, 0, batch_size=16)

===== baseline
Reusing 10972 article ids from /content/drive/MyDrive/BRIO/eval_outputs_full/sample_ids.txt
/content/drive/MyDrive/BRIO/eval_outputs_full/baseline.out already has all 10972 summaries, nothing to generate
Saved 10972 summaries to /content/drive/MyDrive/BRIO/eval_outputs_full/baseline.out
ROUGE-1 0.3937 | ROUGE-2 0.2165 | ROUGE-L 0.3649  (n=10972)
===== brio_1_1_seed1
Reusing 10972 article ids from /content/drive/MyDrive/BRIO/eval_outputs_full/sample_ids.txt
/content/drive/MyDrive/BRIO/eval_outputs_full/brio_1_1_seed1.out already has all 10972 summaries, nothing to generate
Saved 10972 summaries to /content/drive/MyDrive/BRIO/eval_outputs_full/brio_1_1_seed1.out
ROUGE-1 0.4119 | ROUGE-2 0.2274 | ROUGE-L 0.3814  (n=10972)
===== brio_1_1_seed2
Reusing 10972 article ids from /content/drive/MyDrive/BRIO/eval_outputs_full/sample_ids.txt
/content/drive/MyDrive/BRIO/eval_outputs_full/brio_1_1_seed2.out already has all 10972 summaries, nothing to generate
Saved 10972 summaries to 

In [ ]:
# BERTScore for every system on the whole test set. Results are saved in FULL_DIR/bertscore/.
!conda run --no-capture-output -n env python bertscore_eval.py --data_dir "{FULL_DIR}" --systems {SYSTEMS} {PAIRS}

Scoring baseline (10972 summaries)
calculating scores...
computing bert embedding.

computing greedy matching.

done in 39.76 seconds, 275.97 sentences/sec
Scoring brio_1_1_seed1 (10972 summaries)
calculating scores...
computing bert embedding.

computing greedy matching.

done in 41.35 seconds, 265.35 sentences/sec
Scoring brio_1_1_seed2 (10972 summaries)
calculating scores...
computing bert embedding.

computing greedy matching.

done in 40.97 seconds, 267.78 sentences/sec
Scoring brio_1_1_seed3 (10972 summaries)
calculating scores...
computing bert embedding.

computing greedy matching.

done in 40.59 seconds, 270.28 sentences/sec
Scoring brio_10_1_seed1 (10972 summaries)
calculating scores...
computing bert embedding.

computing greedy matching.

done in 42.68 seconds, 257.08 sentences/sec
Scoring brio_10_1_seed2 (10972 summaries)
calculating scores...
computing bert embedding.

computing greedy matching.

done in 38.78 seconds, 282.95 sentences/sec
Scoring brio_10_1_seed3 (10972 s

In [ ]:
# Show the result table (ROUGE and BERTScore side by side, one row per system) and the per-article scores.
import pandas as pd
print(open(os.path.join(FULL_DIR, "bertscore", "bertscore_summary.txt"), encoding="utf-8").read())
scores = pd.read_csv(os.path.join(FULL_DIR, "bertscore", "bertscore.csv"))
scores.head()

BERTScore config: bert-base-multilingual-cased_L9_no-idf_version=0.3.12(hug_trans=4.46.3)
Articles: 10972

                  bertscore_P  bertscore_R  bertscore_F1  rouge1  rouge2  rougeL
system                                                                          
baseline               0.7704       0.7782        0.7741  0.3937  0.2165  0.3649
brio_1_1_seed1         0.7717       0.7858        0.7784  0.4119  0.2274  0.3814
brio_1_1_seed2         0.7703       0.7864        0.7780  0.4095  0.2243  0.3792
brio_1_1_seed3         0.7724       0.7876        0.7797  0.4141  0.2304  0.3831
brio_10_1_seed1        0.7616       0.7835        0.7721  0.3985  0.2152  0.3651
brio_10_1_seed2        0.7678       0.7774        0.7723  0.3953  0.2116  0.3622
brio_10_1_seed3        0.7624       0.7815        0.7715  0.3949  0.2123  0.3619
brio_100_1_seed1       0.7590       0.7792        0.7687  0.3902  0.2068  0.3557
brio_100_1_seed2       0.7661       0.7747        0.7701  0.3899  0.2053  0.3565
br

,id,system,precision,recall,f1
0,0,baseline,0.849274,0.871821,0.860400
1,1,baseline,0.714044,0.765111,0.738696
2,2,baseline,0.780636,0.788293,0.784446
3,3,baseline,0.748610,0.750871,0.749739
4,4,baseline,0.696174,0.701563,0.698858


### Optional: IndoBERT instead of multilingual BERT

The default embedding model is multilingual BERT. To check that the result does not depend on that choice, run the same scoring with IndoBERT. This overwrites the files in `bertscore/`, so copy `bertscore_summary.txt` first if you want to keep both.

In [ ]:
# Same scoring with IndoBERT embeddings (layer 9, the layer bert-score uses for BERT-base models).
!conda run --no-capture-output -n env python bertscore_eval.py --data_dir "{FULL_DIR}" --systems {SYSTEMS} {PAIRS} --model_type indobenchmark/indobert-base-p1 --num_layers 9

In [ ]:
import os
CKPT_DIR = "/content/drive/MyDrive/BRIO/checkpoints"

# BRIO_CKPTS = {
#     "brio_1_1_seed1": CKPT_DIR +  "/BRIO_Liputan6_checkpoint_1-1/8000/model_generation.bin",
#     "brio_1_1_seed2":   CKPT_DIR + "/BRIO_Liputan6_checkpoint_1-1_second_seed/master_1_1/model_generation.bin",
#     "brio_1_1_seed3":   CKPT_DIR + "/BRIO_Liputan6_checkpoint_1-1_third_seed/master_1_1/model_generation.bin",
#     "brio_10_1_seed1":  CKPT_DIR +  "/BRIO_Liputan6_checkpoint_10-1-reduceLR/8000/model_generation.bin",
#     "brio_10_1_seed2":  CKPT_DIR +  "/BRIO_Liputan6_checkpoint_10-1_second_seed/master_1_10/model_generation.bin",
#     "brio_10_1_seed3":  CKPT_DIR +  "/BRIO_Liputan6_checkpoint_10-1_third_seed/master_1_10/model_generation.bin",
#     "brio_100_1_seed1": CKPT_DIR +  "/BRIO_Liputan6_checkpoint_100-1-reduceLR/4000/model_generation.bin",
#     "brio_100_1_seed2": CKPT_DIR +  "/BRIO_Liputan6_checkpoint_100-1-second_seed/master_1_100/model_generation.bin",
#     "brio_100_1_seed3": CKPT_DIR +  "/BRIO_Liputan6_checkpoint_100-1-third_seed/master_1_100/model_generation.bin",
# }
SAMPLE_DIR = "/content/drive/MyDrive/BRIO/eval_outputs"
FULL_DIR = "/content/drive/MyDrive/BRIO/eval_outputs_full"


names = ["baseline"] + list(BRIO_CKPTS)
for name in names:
    lines = open(os.path.join(FULL_DIR, name + ".out"), encoding="utf-8").read().splitlines()
    print("%-18s %.1f words per summary" % (name, sum(len(l.split()) for l in lines) / len(lines)))

In [ ]:
ref = open(os.path.join(FULL_DIR, "sample.target"), encoding="utf-8").read().splitlines()
print("reference %.1f words per summary" % (sum(len(l.split()) for l in ref) / len(ref)))

In [ ]:
import pandas as pd
ids = [int(x) for x in open(os.path.join(FULL_DIR, "sample_ids.txt")).read().split()]
lengths = {n: pd.Series([len(l.split()) for l in open(os.path.join(FULL_DIR, n + ".out"), encoding="utf-8").read().splitlines()], index=ids)
           for n in names}
f1 = pd.read_csv(os.path.join(FULL_DIR, "bertscore", "bertscore.csv")).pivot(index="id", columns="system", values="f1")

for n in names[1:]:
    d = pd.DataFrame({"len_diff": lengths[n] - lengths["baseline"], "f1_diff": f1[n] - f1["baseline"]})
    same = d[d["len_diff"].abs() <= 2]
    print("%-18s all: %+.4f | similar length (%d articles): %+.4f | corr(length diff, F1 diff): %.2f"
          % (n, d["f1_diff"].mean(), len(same), same["f1_diff"].mean(), d["f1_diff"].corr(d["len_diff"])))